In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- locate the file no matter where the notebook is running from ---
here = Path.cwd()
matches = list(here.rglob("PSF_aggregates_databank_Mar_EFO.xlsx"))
if not matches:
    # search upward too, in case cwd is below the repo
    for parent in here.parents:
        matches = list(parent.rglob("PSF_aggregates_databank_Mar_EFO.xlsx"))
        if matches:
            break
assert matches, f"File not found anywhere under {here} or its parents"
path = matches[0]
print("Using:", path)

# --- load the per-cent-of-GDP aggregates tab (real headers are row 3; data from row 4) ---
raw = pd.read_excel(path, sheet_name="Aggregates (per cent of GDP)", header=None)

# cols (0-indexed): year=1, primary balance=11, net debt=23, output gap=32
d = raw.iloc[4:, [1, 11, 23, 32]].copy()
d.columns = ["year", "pb", "debt", "gap"]

def to_year(v):
    s = str(v)
    return int(s[:4]) if s[:4].isdigit() else np.nan

d["year"] = d["year"].apply(to_year)
for c in ["pb", "debt", "gap"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d = d.dropna(subset=["year", "pb", "debt"]).reset_index(drop=True)
d["year"] = d["year"].astype(int)
d = d[d["year"] <= 2024]          # outturn only

# --- sanity check ---
print(f"\nyears: {d.year.min()}–{d.year.max()}   n = {len(d)}")
print("\nsummary:")
print(d.describe().round(2))
print("\nlast rows:")
print(d.tail())

Using: /Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Research/Data/PSF_aggregates_databank_Mar_EFO.xlsx

years: 1974–2024   n = 51

summary:
          year     pb   debt    gap
count    51.00  51.00  51.00  51.00
mean   1999.00  -1.36  50.24  -0.27
std      14.87   3.07  23.61   1.51
min    1974.00 -13.74  21.67  -3.16
25%    1986.50  -2.67  33.76  -1.25
50%    1999.00  -1.17  38.88  -0.10
75%    2011.50   0.69  75.52   0.62
max    2024.00   3.62  95.38   3.27

last rows:
    year         pb       debt       gap
46  2020 -13.739600  95.381028 -0.287472
47  2021  -2.979964  94.313911  1.781415
48  2022  -1.174983  93.214170  1.028379
49  2023  -1.865724  94.200000  0.009025
50  2024  -2.371854  93.200000 -0.410393


In [8]:
import statsmodels.formula.api as smf
from scipy.stats import f as fdist

# lagged debt (Bohn: does the primary balance respond to LAST year's debt?)
d = d.sort_values("year").reset_index(drop=True)
d["debt_lag"] = d["debt"].shift(1)
reg = d.dropna(subset=["debt_lag", "pb", "gap"]).copy()

def bohn(data, label):
    m = smf.ols("pb ~ debt_lag + gap", data=data).fit()
    b, se, p = m.params["debt_lag"], m.bse["debt_lag"], m.pvalues["debt_lag"]
    star = "***" if p < .01 else "**" if p < .05 else "*" if p < .1 else ""
    print(f"{label:28s} n={len(data):3d}  beta={b:+.4f} (se {se:.4f}) p={p:.3f} {star}")
    return m

print("BOHN FISCAL REACTION FUNCTION  (beta>0 = debt-stabilising behaviour)")
print("-" * 72)
m_full = bohn(reg, "Full sample")
bohn(reg[reg.year <= 2009], "Pre-2010")
bohn(reg[reg.year >= 2010], "2010-2024")

# --- Chow test for a break at 2010 ---
reg["post"] = (reg.year >= 2010).astype(int)
m_r = smf.ols("pb ~ debt_lag + gap", data=reg).fit()
m_u = smf.ols("pb ~ debt_lag*post + gap*post", data=reg).fit()
k, n = 3, len(reg)
F = ((m_r.ssr - m_u.ssr) / k) / (m_u.ssr / (n - 2 * k))
print("-" * 72)
print(f"Chow test @2010:  F={F:.2f}  p={1 - fdist.cdf(F, k, n - 2*k):.3f}   "
      f"(high p = NO break)")

# --- shock robustness: drop GFC (2008-09) and COVID (2020-21) ---
print("-" * 72)
shocks = [2008, 2009, 2020, 2021]
bohn(reg[~reg.year.isin(shocks)], "Ex-GFC & COVID")
print("  ^ if beta stays negative & significant here, the result is robust;")
print("    if it flips toward zero/positive, the finding was driven by shocks.")

BOHN FISCAL REACTION FUNCTION  (beta>0 = debt-stabilising behaviour)
------------------------------------------------------------------------
Full sample                  n= 50  beta=-0.0539 (se 0.0156) p=0.001 ***
Pre-2010                     n= 35  beta=-0.0107 (se 0.0526) p=0.841 
2010-2024                    n= 15  beta=-0.0509 (se 0.1661) p=0.764 
------------------------------------------------------------------------
Chow test @2010:  F=0.34  p=0.793   (high p = NO break)
------------------------------------------------------------------------
Ex-GFC & COVID               n= 46  beta=-0.0364 (se 0.0118) p=0.004 ***
  ^ if beta stays negative & significant here, the result is robust;
    if it flips toward zero/positive, the finding was driven by shocks.
